# Water Potability Prediction and Comparative Evaluation Using Machine Learning Models

This notebook trains and compares three supervised machine learning models for predicting whether a water sample is **potable** or **not potable** based on water quality indicators.

**Models compared:**
1. Logistic Regression
2. Decision Tree Classifier
3. Random Forest Classifier

**Target variable:** `Potability`
- `0` = Not Potable
- `1` = Potable

In [ ]:
# 1. Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report, ConfusionMatrixDisplay

import joblib
import json

In [ ]:
# 2. Load dataset
DATA_PATH = 'water_potability.csv'
df = pd.read_csv(DATA_PATH)

df.head()

In [ ]:
# 3. Dataset overview
print('Dataset shape:', df.shape)
print('
Columns:')
print(df.columns.tolist())
print('
Dataset information:')
df.info()

In [ ]:
# 4. Check missing values
missing_values = df.isnull().sum()
missing_values

In [ ]:
# 5. Check target class distribution
class_counts = df['Potability'].value_counts().sort_index()
print(class_counts)

plt.figure(figsize=(6,4))
plt.bar(['Not Potable (0)', 'Potable (1)'], class_counts.values)
plt.title('Target Class Distribution')
plt.ylabel('Number of Samples')
plt.show()

In [ ]:
# 6. Separate features and target
X = df.drop('Potability', axis=1)
y = df['Potability']

feature_names = X.columns.tolist()
feature_names

In [ ]:
# 7. Train-test split
# Stratify keeps the same class proportion in training and testing sets.
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print('Training set:', X_train.shape)
print('Testing set:', X_test.shape)

In [ ]:
# 8. Define ML models
# Missing values are handled using median imputation.
# StandardScaler is used because some models are sensitive to feature scale.
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'),
    'Decision Tree': DecisionTreeClassifier(random_state=42, max_depth=8, class_weight='balanced'),
    'Random Forest': RandomForestClassifier(n_estimators=300, random_state=42, class_weight='balanced')
}

In [ ]:
# 9. Train and evaluate models
results = []
fitted_models = {}

for model_name, model in models.items():
    pipeline = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('model', model)
    ])

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    results.append({
        'Model': model_name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall': recall_score(y_test, y_pred, zero_division=0),
        'F1-score': f1_score(y_test, y_pred, zero_division=0),
        'Weighted F1-score': f1_score(y_test, y_pred, average='weighted', zero_division=0),
        'Confusion Matrix': confusion_matrix(y_test, y_pred)
    })

    fitted_models[model_name] = pipeline

results_df = pd.DataFrame(results)
results_df[['Model', 'Accuracy', 'Precision', 'Recall', 'F1-score', 'Weighted F1-score']]

In [ ]:
# 10. Visual comparison of model performance
metrics_to_plot = ['Accuracy', 'Precision', 'Recall', 'F1-score', 'Weighted F1-score']

for metric in metrics_to_plot:
    plt.figure(figsize=(7,4))
    plt.bar(results_df['Model'], results_df[metric])
    plt.title(f'Model Comparison by {metric}')
    plt.ylabel(metric)
    plt.ylim(0, 1)
    plt.xticks(rotation=15)
    plt.show()

In [ ]:
# 11. Confusion matrix for each model
for item in results:
    model_name = item['Model']
    cm = item['Confusion Matrix']
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Not Potable', 'Potable'])
    disp.plot()
    plt.title(f'Confusion Matrix - {model_name}')
    plt.show()

In [ ]:
# 12. Classification report for each model
for model_name, pipeline in fitted_models.items():
    print('=' * 70)
    print(model_name)
    print('=' * 70)
    y_pred = pipeline.predict(X_test)
    print(classification_report(y_test, y_pred, target_names=['Not Potable', 'Potable'], zero_division=0))

In [ ]:
# 13. Select the best model
# For this project, the best model is selected using Accuracy first, then Weighted F1-score.
# This is suitable because the application needs good overall classification performance.
results_sorted = results_df.sort_values(by=['Accuracy', 'Weighted F1-score'], ascending=False)
best_model_name = results_sorted.iloc[0]['Model']
best_model = fitted_models[best_model_name]

print('Best Model:', best_model_name)
results_sorted[['Model', 'Accuracy', 'Precision', 'Recall', 'F1-score', 'Weighted F1-score']]

In [ ]:
# 14. Save model and metadata for the Streamlit app
joblib.dump(best_model, 'best_model.pkl')

metadata = {
    'best_model': best_model_name,
    'feature_names': feature_names,
    'target_mapping': {'0': 'Not Potable', '1': 'Potable'},
    'selection_basis': 'Highest Accuracy, then Weighted F1-score',
    'results': results_sorted[['Model', 'Accuracy', 'Precision', 'Recall', 'F1-score', 'Weighted F1-score']].to_dict(orient='records')
}

with open('model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=4)

results_sorted[['Model', 'Accuracy', 'Precision', 'Recall', 'F1-score', 'Weighted F1-score']].to_csv('model_results.csv', index=False)

print('Saved best_model.pkl')
print('Saved model_metadata.json')
print('Saved model_results.csv')

In [ ]:
# 15. Test a sample prediction
sample = X_test.iloc[[0]]
prediction = best_model.predict(sample)[0]
probability = best_model.predict_proba(sample)[0]

print('Sample input:')
display(sample)
print('Prediction:', 'Potable' if prediction == 1 else 'Not Potable')
print('Class probabilities:', probability)

## Interpretation

The three models were compared using accuracy, precision, recall, F1-score, and weighted F1-score. The best model was selected based on overall classification performance. The saved model can be integrated into a Streamlit application where users can input water quality measurements and receive a potability prediction.